# Reproduce the direct SGLang layerwise round trip

Run all cells from any directory in an LMCache checkout. This executes the new direct layerwise CUDA round trip together with the adjacent SGLang connector matrix. It fails if any selected case fails or is skipped.

In [ ]:
# SPDX-License-Identifier: Apache-2.0
from pathlib import Path
import os
import subprocess
import sys

import torch

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "lmcache").is_dir()
)
TEST_FILE = ROOT / "tests/v1/test_gpu_connector.py"
assert TEST_FILE.is_file(), TEST_FILE
assert torch.cuda.is_available(), "CUDA is required"
print(
    {
        "root": str(ROOT),
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0),
    }
)

In [ ]:
# SPDX-License-Identifier: Apache-2.0
command = [
    sys.executable,
    "-m",
    "pytest",
    "-q",
    "-s",
    str(TEST_FILE),
    "-k",
    "sglang_layerwise_connector_direct_roundtrip or sglang_connector_with_gpu_and_mla",
]
env = os.environ.copy()
env["PYTHONPATH"] = str(ROOT) + os.pathsep + env.get("PYTHONPATH", "")
print("Running:", " ".join(command))
completed = subprocess.run(
    command,
    cwd=ROOT,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(completed.stdout)
completed.check_returncode()
assert "5 passed" in completed.stdout, (
    "expected the direct regression plus four adjacent cases"
)
print({"status": "passed", "selected_cuda_cases": 5})